In [202]:
import numpy as np
import pandas as pd
import datetime
import yfinance as yf
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

# For importing universal scripts
import sys
import os
# Go up two levels from the subfolder
sys.path.append(os.path.abspath("/.."))
from indicators_returns import final_df #Universal script for indicator set and actuals
import importlib
import indicators_returns
importlib.reload(indicators_returns)
from indicators_returns import final_df
import gc
from sklearn.metrics import (fbeta_score, accuracy_score, f1_score, 
                             confusion_matrix, balanced_accuracy_score, recall_score, matthews_corrcoef, precision_score)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split, StratifiedKFold
from xgboost import XGBClassifier
import math
import pickle

tags = pd.read_csv('../Indicator_Selection_Pipeline/Finalization/tags_cons.csv') 

def extract(ticker, returns, lb, cat_cols_all, windows=[10, 25]):
    
    df = final_df(ticker, returns, lb)
    df = df.iloc[:-101].replace([np.inf, -np.inf], 0)#

    df = df.sort_index(ascending=True)
    # Exponential Moving Average
    ema_cols = {
        f"{col}_EMA{w}": df[col].ewm(span=w, adjust=False).mean()
        for w in windows
        for col in cat_cols_all
    }
    # 3) merge them back into one dict
    new_cols = {**ema_cols}

    # 4) concatenate onto your original df
    df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)    
    df = df.sort_index(ascending=False)
    
    return df

# Start with must include columns (slope)

In [210]:
def print_metrics(metrics):
        for thresh, metric_values in metrics.items():
            print(f"  Threshold {thresh}: {metric_values}")
    
def optimize_ttv(df_indicators, df_predict, thresh, opt, scale_pos_weight, arch, test_size, val_size, perf_size, return_metrics=False):
    
    def train_and_evaluate(model, param_grid, X_train, X_val, y_train, y_val, X_test, y_test, opt, thresh):
        
        # Create a Stratified K-Fold object
        stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

        # Perform Random Search
        random_search = RandomizedSearchCV(
            estimator=model,
            param_distributions=param_grid,  # Corrected from param_grid to param_distributions
            scoring=opt,
            cv=stratified_kfold,
            n_jobs=-1,
            n_iter=40,  # Adjust this based on how many random samples you want to try
            random_state=42  # Ensures reproducibility
        )

        random_search.fit(X_train, y_train, eval_set=[(X_val, y_val)],
        verbose=False)
        best_model = random_search.best_estimator_
        # Predict probabilities
        y_prob = best_model.predict_proba(X_test)
        postot = y_test.sum()
        negtot = len(y_test) - y_test.sum()

        # Evaluate metrics for each threshold
        metrics = {}
        for t in thresh:
            y_pred_thresh = (y_prob[:, 1] > t).astype(int)
            y_pred_thresh[y_prob[:, 0] > t] = 0
            filtered_indices = (y_prob[:, 1] > t) | (y_prob[:, 0] > t)

            if filtered_indices.sum() > 0:
                y_test_valid = y_test[filtered_indices]
                y_pred_valid = y_pred_thresh[filtered_indices]
                posprec = round(precision_score(y_test_valid, y_pred_valid, pos_label=1, zero_division=0), 2)
                negprec = round(precision_score(y_test_valid, y_pred_valid, pos_label=0, zero_division=0), 2)
                poscnt = sum(y_pred_valid == 1)
                negcnt = sum(y_pred_valid == 0)
                metrics[t] = {
                    'TT_Len': len(df_indicators),
                    "PosTot": postot,  
                    "NegTot": negtot,  
                    "PosID": round(posprec * poscnt / postot, 2),
                    'PosPrec': posprec,  
                    "NegID":round(negprec * negcnt / negtot, 2),
                    'NegPrec': negprec,  
                    'PosCnt': poscnt,
                    'NegCnt': negcnt,
                }

        del random_search
        gc.collect()

        return metrics, best_model

    if arch == 'shallow':
        # Shallow
        xgboost_hyperparameters = {
            'scale_pos_weight': [scale_pos_weight],
            'n_estimators': [200, 300],
            'max_depth': [5, 7], 
            'learning_rate': [0.01],
            'subsample': [0.65],
            'colsample_bytree': [0.6], 
            'gamma': [0.2, 0.4],
            'alpha': [0.1, 1], 
            'lambda': [1, 2, 5], 
            'min_child_weight': [12, 15],
            'early_stopping_rounds': [10]
        }

    elif arch == 'moderate':

        # Moderate
        xgboost_hyperparameters = {
            'scale_pos_weight': [scale_pos_weight],
            'n_estimators': [300, 400],
            'max_depth': [7, 9], 
            'learning_rate': [0.01],
            'subsample': [0.65, .75],
            'colsample_bytree': [0.6, 0.7], 
            'gamma': [0.2, 0.3],
            'alpha': [0.1, 1], 
            'lambda': [1, 2, 5], 
            'min_child_weight': [9, 11],
            'early_stopping_rounds': [8]
        }

    else:

        # Deep
        xgboost_hyperparameters = {
            'scale_pos_weight': [scale_pos_weight],
            'n_estimators': [300, 400, 500],
            'max_depth': [8, 10, 12], 
            'learning_rate': [0.01],
            'subsample': [0.75, .85],
            'colsample_bytree': [0.75, 0.85], 
            'gamma': [0.1, 0.2],
            'alpha': [0.1, 1], 
            'lambda': [1, 2, 5], 
            'min_child_weight': [5, 7],
            'early_stopping_rounds': [10]
        }
        
    X_test = df_indicators.iloc[:perf_size].copy()
    y_test = df_predict.iloc[:perf_size].copy()
    df_indicators = df_indicators.iloc[test_size:].copy()
    df_predict = df_predict.iloc[test_size:].copy()

    X_val = df_indicators.iloc[:val_size].copy()
    y_val = df_predict.iloc[:val_size].copy()
    X_train = df_indicators.iloc[val_size:].copy()
    y_train = df_predict.iloc[val_size:].copy()

    # Train and evaluate models
    xg_metrics, best_xg_model = train_and_evaluate(XGBClassifier(random_state=42), xgboost_hyperparameters, X_train, X_val, y_train, y_val, X_test, y_test, opt, thresh)
    #record_validation_metrics(xg_metrics, arch='shallow', horizon=r, model_name=name)
    #savearch(best_xg_model, r, name, arch)
    print_metrics(xg_metrics)

    if return_metrics:
        return xg_metrics, best_xg_model
    else:
        #print_metrics(xg_metrics)
        return best_xg_model
    
def save(best_model, days, model, ticker):
    
    directory = f'../Models/Ensemble_{ticker}'  # Saves to the current working directory
    os.makedirs(directory, exist_ok=True)
    
    with open(os.path.join(directory, str(model)+'_xgboost_'+str(days)+'.pkl'), 'wb') as file_object:
        pickle.dump(best_model, file_object)

    print('Models Saved for '+(ticker)+'_'+str(model)+' '+str(days)+'_'+' Returns')

# Velocity x Category Combo Groupings

In [178]:
raw_all = tags['Indicator'][tags['Type'] == 'Raw'].tolist()
cat_slow = tags[(tags['Type'] == 'Raw') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
cat_moderate = tags[(tags['Type'] == 'Raw') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
cat_fast = tags[(tags['Type'] == 'Raw') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()

windows = [25]
# EMA column‐name lists
cat_slow25      = [f"{col}_EMA{w}" for col in cat_slow     for w in windows] # Only prod model
cat_moderate25  = [f"{col}_EMA{w}" for col in cat_moderate for w in windows]
cat_fast25      = [f"{col}_EMA{w}" for col in cat_fast     for w in windows]

# By category × velocity
duration_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'duration') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
duration_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'duration') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
duration_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'duration') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()

trend_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
trend_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
trend_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()

trend_ratio_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend_ratio') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
trend_ratio_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend_ratio') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
trend_ratio_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend_ratio') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()

volatility_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'volatility') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
volatility_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'volatility') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
volatility_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'volatility') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()

momentum_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'momentum') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
momentum_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'momentum') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
momentum_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'momentum') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()


In [223]:
tickers = ['QQQ']#, 'NVDA', "AAPL", "MSFT", "TSLA", "AMZN", "AVGO", "META", "GOOGL", "COST", "NFLX"]
thresh = [.5]
arch_types = ['deep', 'moderate', 'shallow']
results = []
lb = 8
returns = [2, 3, 4, 5, 6, 7, 8, 9, 10]

df = extract('QQQ', returns, lb, raw_all, windows=[])
returns = [2, 3]

selected_combos = {
    "h_2": {
    "trend_slow_trend_ratio_slow_trend_ratio_fast": trend_slow + trend_ratio_slow + trend_ratio_fast,
    "duration_slow_trend_slow_trend_ratio_slow": duration_slow + trend_slow + trend_ratio_slow,
    "trend_slow_trend_fast_trend_ratio_slow": trend_slow + trend_fast + trend_ratio_slow,
    "trend_slow_trend_ratio_slow_volatility_moderate": trend_slow + trend_ratio_slow + volatility_moderate,
    "duration_slow_duration_moderate_trend_ratio_slow": duration_slow + duration_moderate + trend_ratio_slow, 
    "trend_ratio_slow_trend_ratio_fast_momentum_moderate": trend_ratio_slow + trend_ratio_fast + momentum_moderate, 
    "trend_moderate_trend_ratio_slow_momentum_moderate": trend_moderate + trend_ratio_slow + momentum_moderate, 
    "trend_ratio_slow_trend_ratio_moderate_volatility_moderate": trend_ratio_slow + trend_ratio_moderate + volatility_moderate,
    "duration_slow_trend_ratio_slow_momentum_moderate": duration_slow + trend_ratio_slow + momentum_moderate,
    "trend_slow_trend_moderate_trend_ratio_moderate": trend_slow + trend_moderate + trend_ratio_moderate,
    },
    "h_3": {
    "duration_moderate_trend_slow_trend_moderate": duration_moderate + trend_slow + trend_moderate,
    "duration_slow_trend_slow_trend_ratio_slow": duration_slow + trend_slow + trend_ratio_slow, 
    "trend_slow_trend_ratio_slow_momentum_slow": trend_slow + trend_ratio_slow + momentum_slow,
    "duration_slow_trend_slow_trend_ratio_moderate": duration_slow + trend_slow + trend_ratio_moderate,
    "trend_slow_trend_ratio_slow_trend_ratio_moderate": trend_slow + trend_ratio_slow + trend_ratio_moderate,
    "trend_slow_trend_moderate_trend_ratio_slow": trend_slow + trend_moderate + trend_ratio_slow,
    "duration_fast_trend_slow_trend_ratio_moderate": duration_fast + trend_slow + trend_ratio_moderate,
    "duration_slow_trend_ratio_slow_trend_ratio_moderate": duration_slow + trend_ratio_slow + trend_ratio_moderate,
    "duration_slow_trend_moderate_trend_ratio_slow": duration_slow + trend_moderate + trend_ratio_slow,
    "trend_slow_trend_ratio_moderate_volatility_moderate": trend_slow + trend_ratio_moderate + volatility_moderate,
    },
    "h_4": {
    'duration_slow_duration_moderate_trend_moderate_trend_ratio_slow_momentum_moderate': duration_slow + duration_moderate + trend_moderate + trend_ratio_slow + momentum_moderate,
    'duration_slow_duration_fast_trend_moderate_trend_ratio_slow_momentum_moderate': duration_slow + duration_fast + trend_moderate + trend_ratio_slow + momentum_moderate,
    'duration_slow_duration_fast_trend_slow_trend_moderate_trend_ratio_moderate': duration_slow + duration_fast + trend_slow + trend_moderate + trend_ratio_moderate,
    'duration_moderate_trend_slow_trend_ratio_slow_trend_ratio_fast_momentum_moderate': duration_moderate + trend_slow + trend_ratio_slow + trend_ratio_fast + momentum_moderate,
    'duration_moderate_trend_slow_trend_ratio_slow_trend_ratio_fast_momentum_fast': duration_moderate + trend_slow + trend_ratio_slow + trend_ratio_fast + momentum_fast,
    'duration_slow_trend_slow_trend_moderate_trend_ratio_slow_momentum_moderate': duration_slow + trend_slow + trend_moderate + trend_ratio_slow + momentum_moderate,
    'duration_slow_duration_moderate_trend_slow_trend_ratio_slow_trend_ratio_fast': duration_slow + duration_moderate + trend_slow + trend_ratio_slow + trend_ratio_fast,
    'duration_slow_duration_fast_trend_slow_trend_ratio_slow_trend_ratio_fast': duration_slow + duration_fast + trend_slow + trend_ratio_slow + trend_ratio_fast,
    'trend_slow_trend_ratio_slow_trend_ratio_fast_momentum_moderate_momentum_fast': trend_slow + trend_ratio_slow + trend_ratio_fast + momentum_moderate + momentum_fast,
    'trend_slow_trend_ratio_slow_momentum_fast': trend_slow + trend_ratio_slow + momentum_fast,
    },
    "h_5": {
    'duration_slow_duration_moderate_trend_moderate_trend_ratio_slow_momentum_moderate': duration_slow + duration_moderate + trend_moderate + trend_ratio_slow + momentum_moderate,
    'duration_slow_duration_fast_trend_moderate_trend_ratio_slow_momentum_moderate': duration_slow + duration_fast + trend_moderate + trend_ratio_slow + momentum_moderate,
    'duration_slow_duration_fast_trend_slow_trend_moderate_trend_ratio_moderate': duration_slow + duration_fast + trend_slow + trend_moderate + trend_ratio_moderate,
    'duration_moderate_trend_slow_trend_ratio_slow_trend_ratio_fast_momentum_moderate': duration_moderate + trend_slow + trend_ratio_slow + trend_ratio_fast + momentum_moderate,
    'duration_moderate_trend_slow_trend_ratio_slow_trend_ratio_fast_momentum_fast': duration_moderate + trend_slow + trend_ratio_slow + trend_ratio_fast + momentum_fast,
    'duration_slow_trend_slow_trend_moderate_trend_ratio_slow_momentum_moderate': duration_slow + trend_slow + trend_moderate + trend_ratio_slow + momentum_moderate,
    'duration_slow_duration_moderate_trend_slow_trend_ratio_slow_trend_ratio_fast': duration_slow + duration_moderate + trend_slow + trend_ratio_slow + trend_ratio_fast,
    'trend_slow_trend_ratio_moderate_volatility_moderate': trend_slow + trend_ratio_moderate + volatility_moderate,
    'duration_moderate_trend_slow_trend_ratio_moderate_trend_ratio_fast_momentum_fast': duration_moderate + trend_slow + trend_ratio_moderate + trend_ratio_fast + momentum_fast,
    'duration_slow_trend_slow_trend_fast_trend_ratio_moderate_volatility_moderate': duration_slow + trend_slow + trend_fast + trend_ratio_moderate + volatility_moderate,
    },
    "h_6": {
    'duration_slow_duration_moderate_trend_moderate_trend_ratio_slow_momentum_moderate': duration_slow + duration_moderate + trend_moderate + trend_ratio_slow + momentum_moderate,
    'duration_slow_duration_fast_trend_moderate_trend_ratio_slow_momentum_moderate': duration_slow + duration_fast + trend_moderate + trend_ratio_slow + momentum_moderate,
    'duration_slow_duration_fast_trend_slow_trend_moderate_trend_ratio_moderate': duration_slow + duration_fast + trend_slow + trend_moderate + trend_ratio_moderate,
    'duration_moderate_trend_slow_trend_ratio_slow_trend_ratio_fast_momentum_moderate': duration_moderate + trend_slow + trend_ratio_slow + trend_ratio_fast + momentum_moderate,
    'duration_moderate_trend_slow_trend_ratio_slow_trend_ratio_fast_momentum_fast': duration_moderate + trend_slow + trend_ratio_slow + trend_ratio_fast + momentum_fast,
    'duration_slow_trend_slow_trend_moderate_trend_ratio_slow_momentum_moderate': duration_slow + trend_slow + trend_moderate + trend_ratio_slow + momentum_moderate,
    'duration_slow_duration_moderate_trend_slow_trend_ratio_slow_trend_ratio_fast': duration_slow + duration_moderate + trend_slow + trend_ratio_slow + trend_ratio_fast,
    'trend_slow_trend_ratio_slow_momentum_fast': trend_slow + trend_ratio_slow + momentum_fast,
    'trend_slow_trend_ratio_slow_trend_ratio_fast_momentum_moderate_momentum_fast': trend_slow + trend_ratio_slow + trend_ratio_fast + momentum_moderate + momentum_fast,
    'trend_moderate_trend_fast_trend_ratio_slow_trend_ratio_moderate_momentum_slow': trend_moderate + trend_fast + trend_ratio_slow + trend_ratio_moderate + momentum_slow,
    },
    "h_8": {
    'duration_slow_duration_moderate_trend_slow_trend_ratio_slow_volatility_slow': duration_slow + duration_moderate + trend_slow + trend_ratio_slow + volatility_slow,
    'trend_moderate_trend_ratio_slow_volatility_slow_momentum_slow_momentum_moderate': trend_moderate + trend_ratio_slow + volatility_slow + momentum_slow + momentum_moderate,
    'duration_slow_duration_moderate_trend_slow_trend_fast_trend_ratio_moderate': duration_slow + duration_moderate + trend_slow + trend_fast + trend_ratio_moderate,
    'duration_slow_trend_moderate_trend_ratio_slow_trend_ratio_fast_momentum_moderate': duration_slow + trend_moderate + trend_ratio_slow + trend_ratio_fast + momentum_moderate,
    'duration_slow_trend_moderate_trend_ratio_slow_momentum_slow_momentum_moderate': duration_slow + trend_moderate + trend_ratio_slow + momentum_slow + momentum_moderate,
    'duration_moderate_trend_fast_trend_ratio_moderate_trend_ratio_fast_momentum_moderate': duration_moderate + trend_fast + trend_ratio_moderate + trend_ratio_fast + momentum_moderate,
    'duration_moderate_trend_moderate_trend_ratio_slow_trend_ratio_fast_volatility_fast': duration_moderate + trend_moderate + trend_ratio_slow + trend_ratio_fast + volatility_fast,
    'duration_moderate_trend_moderate_trend_ratio_slow_trend_ratio_fast_momentum_slow': duration_moderate + trend_moderate + trend_ratio_slow + trend_ratio_fast + momentum_slow,
    'duration_moderate_trend_fast_trend_ratio_moderate_trend_ratio_fast_momentum_moderate': duration_moderate + trend_fast + trend_ratio_moderate + trend_ratio_fast + momentum_moderate,
    'trend_slow_trend_moderate_trend_ratio_moderate_trend_ratio_fast_volatility_fast': trend_slow + trend_moderate + trend_ratio_moderate + trend_ratio_fast + volatility_fast,
    },
    "h_10": {
    'duration_slow_duration_moderate_trend_slow_trend_ratio_slow_volatility_slow': duration_slow + duration_moderate + trend_slow + trend_ratio_slow + volatility_slow,
    'trend_moderate_trend_ratio_slow_volatility_slow_momentum_slow_momentum_moderate': trend_moderate + trend_ratio_slow + volatility_slow + momentum_slow + momentum_moderate,
    'duration_slow_duration_moderate_trend_slow_trend_fast_trend_ratio_moderate': duration_slow + duration_moderate + trend_slow + trend_fast + trend_ratio_moderate,
    'duration_slow_trend_moderate_trend_ratio_slow_trend_ratio_fast_momentum_moderate': duration_slow + trend_moderate + trend_ratio_slow + trend_ratio_fast + momentum_moderate,
    'duration_slow_trend_moderate_trend_ratio_slow_momentum_slow_momentum_moderate': duration_slow + trend_moderate + trend_ratio_slow + momentum_slow + momentum_moderate,
    'duration_slow_trend_ratio_slow_trend_ratio_moderate_momentum_slow_momentum_moderate': duration_slow + trend_ratio_slow + trend_ratio_moderate + momentum_slow + momentum_moderate,
    'duration_moderate_trend_moderate_trend_ratio_slow_trend_ratio_fast_volatility_fast': duration_moderate + trend_moderate + trend_ratio_slow + trend_ratio_fast + volatility_fast,
    'duration_moderate_trend_moderate_trend_ratio_slow_trend_ratio_fast_momentum_slow': duration_moderate + trend_moderate + trend_ratio_slow + trend_ratio_fast + momentum_slow,
    'duration_moderate_trend_fast_trend_ratio_moderate_trend_ratio_fast_momentum_moderate': duration_moderate + trend_fast + trend_ratio_moderate + trend_ratio_fast + momentum_moderate,
    'trend_slow_trend_moderate_trend_ratio_moderate_trend_ratio_fast_volatility_fast': trend_slow + trend_moderate + trend_ratio_moderate + trend_ratio_fast + volatility_fast,
    },
    "h_15_20": {
    'duration_fast_trend_ratio_slow_trend_ratio_moderate_volatility_moderate_momentum_slow': duration_fast + trend_ratio_slow + trend_ratio_moderate + volatility_moderate + momentum_slow,
    'duration_moderate_duration_fast_trend_moderate_trend_ratio_moderate_volatility_moderate': duration_moderate + duration_fast + trend_moderate + trend_ratio_moderate + volatility_moderate,
    'duration_moderate_duration_fast_trend_fast_trend_ratio_moderate_volatility_moderate': duration_moderate + duration_fast + trend_fast + trend_ratio_moderate + volatility_moderate,
    'trend_moderate_trend_ratio_slow_momentum_moderate': trend_moderate + trend_ratio_slow + momentum_moderate,
    'duration_slow_trend_moderate_trend_ratio_moderate_momentum_slow_momentum_moderate': duration_slow + trend_moderate + trend_ratio_moderate + momentum_slow + momentum_moderate,
    'duration_moderate_duration_fast_trend_ratio_slow_trend_ratio_moderate_volatility_moderate': duration_moderate + duration_fast + trend_ratio_slow + trend_ratio_moderate + volatility_moderate,
    'duration_moderate_trend_moderate_trend_fast_trend_ratio_slow_momentum_slow': duration_moderate + trend_moderate + trend_fast + trend_ratio_slow + momentum_slow,
    'duration_moderate_trend_ratio_slow_momentum_moderate': duration_moderate + trend_ratio_slow + momentum_moderate,
    'duration_fast_trend_slow_trend_ratio_slow_trend_ratio_moderate_volatility_moderate': duration_fast + trend_slow + trend_ratio_slow + trend_ratio_moderate + volatility_moderate,
    'duration_fast_trend_moderate_trend_ratio_moderate_volatility_moderate_momentum_slow': duration_fast + trend_moderate + trend_ratio_moderate + volatility_moderate + momentum_slow,
    },
    "h_25_30": {
    'cat_slow': cat_slow25,
    'duration_moderate_trend_moderate_trend_ratio_moderate_momentum_slow_momentum_moderate': duration_moderate + trend_moderate + trend_ratio_moderate + momentum_slow + momentum_moderate,
    'duration_moderate_trend_moderate_trend_fast_trend_ratio_moderate_trend_ratio_fast': duration_moderate + trend_moderate + trend_fast + trend_ratio_moderate + trend_ratio_fast,
    'duration_moderate_trend_fast_trend_ratio_moderate_trend_ratio_fast_momentum_moderate': duration_moderate + trend_fast + trend_ratio_moderate + trend_ratio_fast + momentum_moderate,
    'duration_slow_duration_moderate_trend_moderate_trend_ratio_fast_momentum_slow': duration_slow + duration_moderate + trend_moderate + trend_ratio_fast + momentum_slow,
    'duration_slow_duration_moderate_trend_fast_trend_ratio_moderate_momentum_slow': duration_slow + duration_moderate + trend_fast + trend_ratio_moderate + momentum_slow,
    'duration_slow_trend_slow_trend_ratio_slow_trend_ratio_moderate_momentum_slow': duration_slow + trend_slow + trend_ratio_slow + trend_ratio_moderate + momentum_slow,
    'trend_slow_trend_ratio_slow_trend_ratio_moderate_volatility_moderate_momentum_slow': trend_slow + trend_ratio_slow + trend_ratio_moderate + volatility_moderate + momentum_slow,
    'duration_moderate_trend_moderate_trend_ratio_moderate_volatility_fast_momentum_moderate': duration_moderate + trend_moderate + trend_ratio_moderate + volatility_fast + momentum_moderate,
    'duration_slow_duration_moderate_trend_slow_trend_ratio_slow_trend_ratio_fast': duration_slow + duration_moderate + trend_slow + trend_ratio_slow + trend_ratio_fast,
    }
}

def get_model_set(horizon):
    if horizon in [2]:
        key = "h_2"
    elif horizon in [3]:
        key = "h_3"
    elif horizon in [4]:
        key = "h_4"
    elif horizon in [5]:
        key = "h_5" 
    elif horizon in [6]:
        key = "h_6"
    elif horizon in [8]:
        key = "h_8"
    elif horizon in [10]:
        key = "h_10"
    elif horizon in [15, 20]:
        key = "h_15_20"
    elif horizon in [25, 30]:
        key = "h_25_30"
    elif horizon in [45, 60, 75, 90]:
        key = "h_45_90"
    else:
        raise ValueError(f"Unsupported horizon: {horizon}")
    
    return selected_combos[key]

for ticker in tickers:

    for r in returns:
            
        selected_models = get_model_set(r)
        
        # all index-combos of length r
        for name, cols in selected_models.items():

            df_ph = df.copy()
            df_ph = df_ph.iloc[r:].copy()
            return_col = f"Return_{r}"
            model_key = f"QQQ_{r}"
            counts = df_ph[return_col].value_counts()
            neg = counts.get(0, 0)
            pos = counts.get(1, 1)  # prevent division by zero
            scale_pos_weight = neg / pos

            for arch in arch_types:

                # Choose evaluation metric
                opt = 'matthews_corrcoef'

                # Combine features with return column, drop missing
                cols += ['Close_slope10', 'Close_slope25', 'Close_slope50']
                cols = list(dict.fromkeys(cols))

                used_cols = cols + [return_col]
                df_model = df_ph[used_cols].dropna()
                #print(df_ph['Date'].iloc[0])
                
                df_indicators = df_model[cols]
                df_indicators = df_indicators.replace([np.inf, -np.inf], 0)
                df_predict = df_model[return_col]
                
                print(f"Results for {name} | {arch} | {ticker}_{r}")
                val_size = 300
                test_size = 100
                perf_size = 100
                #xg_metrics, best_xg_model = optimize_tests(df_indicators, df_predict, thresh, opt, depth, scale_pos_weight, min_child_weight, r, name, arch, date, return_metrics=True)
                best_xg_model = optimize_ttv(df_indicators, df_predict, thresh, opt, scale_pos_weight, arch, test_size, val_size, perf_size, return_metrics=False)
                save(best_xg_model, r, name, ticker)
                #metrics = list(xg_metrics.values())[0]
                # build one flat record
                #row = {'name': name, 'lb': lb, 'arch': arch, 'ticker': ticker, 'horizon': r, 'date': date, **metrics}
                #results.append(row)
                print('---------------------------')

Results for trend_slow_trend_ratio_slow_trend_ratio_fast | deep | QQQ_2
  Threshold 0.5: {'TT_Len': 1656, 'PosTot': 55, 'NegTot': 45, 'PosID': 0.78, 'PosPrec': 0.65, 'NegID': 0.49, 'NegPrec': 0.65, 'PosCnt': 66, 'NegCnt': 34}
Models Saved for QQQ_trend_slow_trend_ratio_slow_trend_ratio_fast 2_ Returns
---------------------------
Results for trend_slow_trend_ratio_slow_trend_ratio_fast | moderate | QQQ_2
  Threshold 0.5: {'TT_Len': 1656, 'PosTot': 55, 'NegTot': 45, 'PosID': 0.77, 'PosPrec': 0.63, 'NegID': 0.45, 'NegPrec': 0.61, 'PosCnt': 67, 'NegCnt': 33}
Models Saved for QQQ_trend_slow_trend_ratio_slow_trend_ratio_fast 2_ Returns
---------------------------
Results for trend_slow_trend_ratio_slow_trend_ratio_fast | shallow | QQQ_2
  Threshold 0.5: {'TT_Len': 1656, 'PosTot': 55, 'NegTot': 45, 'PosID': 0.83, 'PosPrec': 0.61, 'NegID': 0.36, 'NegPrec': 0.64, 'PosCnt': 75, 'NegCnt': 25}
Models Saved for QQQ_trend_slow_trend_ratio_slow_trend_ratio_fast 2_ Returns
---------------------------


KeyboardInterrupt: 

In [200]:
import os
# point this at an existing directory in your project
os.chdir('/Users/brettchase/Documents/Fracturion/Feature_Set_Construction_Testing')

In [197]:
model_dir = f'../Models/Ensemble_{ticker}'
returns = [2, 3, 4, 5, 6, 8, 10]

# Thresholds for prediction decision
upper = thresh[0]
lower = 1 - upper

# Placeholder for final predictions
combined_predictions = {}

# Iterate through each return horizon
for r in returns:

    return_col = f"Return_{r}"

    df_ph = df.copy()

    # Create base prediction matrix with Date and Close
    pred_matrix = df_ph[['Date', 'Close']].copy()

    for arch in arch_types:
    
        selected_models = get_model_set(r)
            
        # all index-combos of length r
        for name, cols in selected_models.items():

            df_features = df_ph[cols].copy()
            df_features = df_features.replace([np.inf, -np.inf], 0)

            # XGBoost prediction
            xgb_path = os.path.join(model_dir, f"{name}_xgboost_{r}.pkl")
            if os.path.exists(xgb_path):
                with open(xgb_path, 'rb') as f:
                    model_xgb = pickle.load(f)
                prob_xgb = model_xgb.predict_proba(df_features)[:, 1]
                col_name_xgb = f"{name}_xgb_{r}{arch}"
                pred_matrix[col_name_xgb] = np.where(
                    prob_xgb > upper, prob_xgb,
                    np.where(prob_xgb < lower, prob_xgb - 1, 0)
                )

            # Final summed prediction across models
            pred_cols = [col for col in pred_matrix.columns if col not in ['Date', 'Close'] and not col.startswith('sum_')]
            pred_matrix[f"sum_{r}"] = pred_matrix[pred_cols].sum(axis=1)

            combined_predictions[r] = pred_matrix

final_df = None

for r, df_preds in combined_predictions.items():
    cols_to_merge = [col for col in df_preds.columns if col not in ['Date', 'Close']]
    df_preds_renamed = df_preds[['Date', 'Close'] + cols_to_merge].copy()

    if final_df is None:
        final_df = df_preds_renamed
    else:
        final_df = pd.merge(final_df, df_preds_renamed, on=['Date', 'Close'], how='outer')

sub_model_df = final_df.copy()
sub_model_df[['Date', 'Close', 'sum_2', 'sum_3', 'sum_4', 'sum_5', 'sum_6', 'sum_8', 'sum_10']].sort_index(ascending=False).head(30)

,Date,Close,sum_2,sum_3,sum_4,sum_5,sum_6,sum_8,sum_10
1757,2025-07-11,554.200012,4.345706,4.323523,5.991677,5.968352,6.455901,5.549220,6.272817
1756,2025-07-10,555.450012,4.324816,3.313599,5.851936,5.957367,5.156468,5.384417,6.271466
1755,2025-07-09,556.250000,4.409517,3.335701,5.796269,4.725129,6.143712,5.349485,6.369278
1754,2025-07-08,552.340027,4.528562,4.450614,6.140474,6.306348,6.552043,5.639288,6.547789
1753,2025-07-07,552.030029,4.473497,4.432172,6.102396,6.278294,6.526916,5.516853,6.819955
1752,2025-07-03,556.219971,4.411863,4.457725,5.884731,5.697215,6.318323,5.603893,6.683833
1751,2025-07-02,550.799988,4.431788,4.415898,5.953125,5.994232,6.519002,5.777844,6.876224
1750,2025-07-01,546.989990,4.559909,4.410561,6.015203,6.266729,6.802437,5.828635,6.990273
1749,2025-06-30,551.640015,4.382397,4.450529,5.878155,5.809347,6.021566,5.730403,6.878254
1748,2025-06-27,548.090027,4.384579,4.415240,5.912419,5.856032,6.384503,5.717207,7.073558


In [161]:
# BEFORE any loops:
feature_importance_records = []

for r in returns:
    return_col = f"Return_{r}"
    df_ph = df.copy()
    pred_matrix = df_ph[['Date','Close']].copy()

    for arch in arch_types:
        selected_models = get_model_set(r)
        
        for name, cols in selected_models.items():
            # prepare features
            df_features = df_ph[cols].replace([np.inf, -np.inf], 0)

            # load model if it exists
            xgb_path = os.path.join(model_dir, f"{name}_xgboost_{r}.pkl")
            if not os.path.exists(xgb_path):
                continue
            with open(xgb_path,'rb') as f:
                model_xgb = pickle.load(f)

            # 1) record the top-5 importances
            importances = model_xgb.feature_importances_
            feat_imp_pairs = list(zip(cols, importances))
            for feature, imp in sorted(feat_imp_pairs, key=lambda x: x[1], reverse=True)[:5]:
                feature_importance_records.append({
                    'horizon':   r,
                    'arch':      arch,
                    'model':     name,
                    'feature':   feature,
                    'importance': imp
                })

            # 2) your existing prediction code
            prob_xgb     = model_xgb.predict_proba(df_features)[:,1]
            col_name_xgb = f"{name}_xgb_{r}{arch}"
            pred_matrix[col_name_xgb] = np.where(
                prob_xgb > upper, prob_xgb,
                np.where(prob_xgb < lower, prob_xgb - 1, 0)
            )

# AFTER all loops:
fi_df = pd.DataFrame(feature_importance_records)

In [162]:
fi_df.to_csv('fi.csv')

In [163]:
def performance_metrics(metrics_df, returns):

    metrics_df = metrics_df.sort_index(ascending=True)

    def add_column_based_on_future_value(df, days):

        future_return = (df['Close'].shift(-days) - df['Close']) / df['Close']

        if days >= 100:
            df[f'Return_{days}'] = np.where(
                future_return > 0.001, 1,
                np.where(future_return < -0.001, 0, np.nan)
            )
        else:
            df[f'Return_{days}'] = (future_return > 0).astype(int)

        return df

    # Apply return logic for each target horizon
    for r in returns: 
        new_df = add_column_based_on_future_value(metrics_df, r)

    return new_df

performance_df = performance_metrics(sub_model_df.round(3), returns)

In [90]:
horizons = returns
# Dictionary to store the filtered DataFrames
horizon_dfs = {}

# Loop through each horizon
for h in horizons:
    # Collect relevant columns
    cols = ['Date', 'Close'] + [
        col for col in performance_df.columns 
        if col.endswith(f'_xgb_{h}deep') or col == f'sum_{h}' or col == f'Return_{h}'
    ]
    
    # Create and store the filtered DataFrame
    horizon_dfs[h] = performance_df[cols].copy()

def convert_signed_to_prob(p):
    return p if p >= 0 else 1 + p

In [111]:
import pandas as pd

# never wrap a wide DataFrame across multiple lines
pd.set_option('display.expand_frame_repr', False)
# show all columns
pd.set_option('display.max_columns', None)
# pretend the display is super wide
pd.set_option('display.width', 10000)
       

In [121]:
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve, auc,
    precision_recall_curve, average_precision_score,
    f1_score, accuracy_score
)

for h in horizons:
    df = horizon_dfs[h].sort_values('Date', ascending=False).iloc[h:h+50, :].copy()
    """
    plt.figure(figsize=(10, 6))
    for col in df.columns:
        if f'sum_{h}' in col:
            filtered = df[df[col] != 0]
            probs = filtered[col].apply(convert_signed_to_prob)
            targets = filtered[f'Return_{h}']
            fpr, tpr, _ = roc_curve(targets, probs)
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, label=f'{col} (AUC = {roc_auc:.2f})')

    plt.plot([0, 1], [0, 1], 'k--')
    plt.title(f'ROC Curve for All Models - Horizon {h}')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    #plt.legend()
    plt.grid()
    plt.show()

    # --- Calibration Curve ---
    plt.figure(figsize=(10, 6))
    for col in df.columns:
        if f'_xgb_{h}' in col:
            filtered = df[df[col] != 0]
            probs = filtered[col].apply(convert_signed_to_prob)
            targets = filtered[f'Return_{h}']
            prob_true, prob_pred = calibration_curve(targets, probs, n_bins=33)
            plt.plot(prob_pred, prob_true, marker='o', label=col)

    plt.plot([0, 1], [0, 1], 'k--')
    plt.title(f'Calibration Curve - Horizon {h}')
    plt.xlabel('Predicted Probability')
    plt.ylabel('Actual Frequency')
    #plt.legend()
    plt.grid()
    plt.show()
    """

    records = []
    actual_pos = df[f'Return_{h}'].sum()
    actual_neg = len(df[f'Return_{h}']) - actual_pos
    for col in df.columns:
        if f'_xgb_{h}' in col:
            # only keep the rows that actually got a prediction
            filtered = df[df[col] != 0]
            # convert your raw score to a probability
            probs = filtered[col].apply(convert_signed_to_prob)
            # binarize at 0.5
            y_pred = (probs >= 0.5).astype(int)
            y_true = filtered[f'Return_{h}']
            
            # compute the matrix
            cm = confusion_matrix(y_true, y_pred,labels=[0,1])
            tn, fp, fn, tp = cm.ravel()
            records.append({
                'model': col,
                'act_pos': actual_pos,
                'act_neg': actual_neg,
                'pos_acc': f"{(tp/actual_pos):.1%}" if actual_pos>0 else "n/a",
                'neg_acc': f"{(tn/actual_neg):.1%}" if actual_neg>0 else "n/a",
                'tp': tp,
                'fp': fp,
                'tn': tn,
                'fn': fn,
            })

    # make your summary table
    cm_df = pd.DataFrame(records)
    print(cm_df.to_string())
    print(' ')

                                                                                         model  act_pos  act_neg pos_acc neg_acc  tp  fp  tn  fn
0  duration_slow_duration_moderate_trend_moderate_trend_ratio_slow_momentum_moderate_xgb_4deep       40       10   72.5%   40.0%  29   6   4  11
1      duration_slow_duration_fast_trend_moderate_trend_ratio_slow_momentum_moderate_xgb_4deep       40       10   87.5%   20.0%  35   8   2   5
2         duration_slow_duration_fast_trend_slow_trend_moderate_trend_ratio_moderate_xgb_4deep       40       10   82.5%   30.0%  33   7   3   7
3   duration_moderate_trend_slow_trend_ratio_slow_trend_ratio_fast_momentum_moderate_xgb_4deep       40       10   87.5%   30.0%  35   7   3   5
4       duration_moderate_trend_slow_trend_ratio_slow_trend_ratio_fast_momentum_fast_xgb_4deep       40       10   87.5%   30.0%  35   7   3   5
5         duration_slow_trend_slow_trend_moderate_trend_ratio_slow_momentum_moderate_xgb_4deep       40       10   85.0%   30.0%  